# Fase de Preprocesamiento de Datos para opciones 

Se utilizará la clase `PreprocesamientoOpciones` para transformar el archivo bruto `option_chains_all.csv`.
* **Limpieza:** Normalización de fechas a formato UTC y eliminación de registros sin cotización (*bid/ask*).
* **Identificación ATM:** Localización de contratos cuyo *strike* sea el más cercano al $S_0$ del 21-01-2026.
* **Filtrado de Liquidez:** Selección de contratos con *Open Interest* activo y bajo desfase temporal en el último intercambio.

In [1]:
import pickle
import pandas as pd
from PreprocesamientoOpciones import PreprocesamientoOpciones


with open('../data/parametros_activos.pkl', 'rb') as f:
    resumen_activos = pickle.load(f)


S0_dict = resumen_activos['S0'].to_dict()


df_opt_raw = pd.read_csv("../data/option_chains_all.csv")


proc_opciones = PreprocesamientoOpciones(df_opt_raw, S0_dict)


df_final = proc_opciones.ejecutar_pipeline(semanas=[21, 52], oi_min=10)


import os; os.makedirs('data', exist_ok=True)
df_final.to_pickle('data/opciones_finales.pkl')

print("\n--- Resumen de Datos Listos para Modelar ---")
print(proc_opciones.resumen())

Iniciando Pipeline de Opciones...
✅ Pipeline finalizado. 8 contratos listos.

--- Resumen de Datos Listos para Modelar ---
{'num_opciones': 8, 'tickers': ['AAPL', 'NVDA', 'INTC', 'COST', 'WBD', 'TSLA', 'META', 'NFLX'], 'vencimientos': ['2026-08-21', '2027-03-19']}


In [2]:
df_final.to_csv('../data/opciones_preprocesadas.csv', index=False)